# Single User Downlink Power Optimization

This notebook follows the single-user candidate-space walkthrough and then asks a different question. Once the shared single-user model admits many feasible PHY allocations, how does the power-optimal choice move as user demand changes?

The answer is developed through two complementary studies built on the same single-user engine: a rate sweep at a fixed 400 m link distance, and a distance sweep from 50 m to 500 m for a fixed 250 Mbps requirement. The notebook is therefore not a verification log. It is a reader-facing study of when the optimal operating point changes, which PA stays competitive, and which resource choices move with the demand.

The early sections define the study setup and summarize the shared search domain compactly. The later sections then use grouped plots to tell the optimization story in stages rather than presenting every diagnostic at once.


## 1. Setup and Notebook Scope

We keep the notebook close to the cleaned candidate-space walkthrough in tone. The first job is to define the two study sweeps, pin down one reference scenario, and make the shared search domain visible without flooding the reader with raw arrays or debug printouts.

That reference scenario is only structural context. It is used to summarize the fixed radio assumptions and the admissible candidate space before the full sweeps are executed.


In [ ]:
import os
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, str((Path.cwd() / "src").resolve()))

NOTEBOOK_OUTER_PARALLEL = True
NOTEBOOK_MAX_WORKERS = max(1, os.cpu_count() or 1)

from single_user_parameter_space import (
    build_single_user_pa_curve_table,
    build_single_user_scenario,
    run_distance_study,
    run_rate_study,
    summarize_single_user_scenario,
)


## 2. Study Definitions

The notebook compares two demand sweeps that share the same underlying radio assumptions and PA catalog.

- The **rate study** fixes the link distance at 400 m and asks how the optimal solution changes as the required average downlink rate increases.
- The **distance study** fixes the required rate at 250 Mbps and asks how the optimal solution changes as the user moves farther from the transmitter.

A low-demand reference scenario is also created here. It is not meant to stand in for the full study. Its job is to expose the common deployment and search-space context that both sweeps inherit.


In [2]:
rate_required_targets_bps = np.linspace(1e6, 300e6, 100)
distance_values_m = np.arange(50.0, 500.0 + 25.0, 25.0).tolist()

rate_study_config = {
    "study_name": "rate",
    "distance_m": 400.0,
    "rate_targets_bps": rate_required_targets_bps.tolist(),
}

distance_study_config = {
    "study_name": "distance",
    "distance_values_m": distance_values_m,
    "rate_target_bps": 250e6,
}

reference_scenario = build_single_user_scenario(
    distance_m=float(rate_study_config["distance_m"]),
    required_rate_bps=float(np.min(rate_required_targets_bps)),
)
reference_search_views = summarize_single_user_scenario(reference_scenario)
reference_candidate_space_view = reference_search_views["candidate_space_view"].copy()
pa_characteristics = reference_search_views["pa_characteristics"].copy()
reference_candidate_space_row = reference_candidate_space_view.iloc[0]

study_definition_view = pd.DataFrame(
    [
        {
            "study": "Rate sweep",
            "varied quantity": "Required rate",
            "fixed quantity": f"Distance = {float(rate_study_config['distance_m']):.0f} m",
            "domain": (
                f"{float(np.min(rate_required_targets_bps)) / 1e6:.0f} to "
                f"{float(np.max(rate_required_targets_bps)) / 1e6:.0f} Mbps"
            ),
            "sampling": (
                f"{len(rate_required_targets_bps)} points, "
                f"step {float(rate_required_targets_bps[1] - rate_required_targets_bps[0]) / 1e6:.1f} Mbps"
            ),
        },
        {
            "study": "Distance sweep",
            "varied quantity": "Link distance",
            "fixed quantity": f"Required rate = {float(distance_study_config['rate_target_bps']) / 1e6:.0f} Mbps",
            "domain": (
                f"{float(min(distance_values_m)):.0f} to "
                f"{float(max(distance_values_m)):.0f} m"
            ),
            "sampling": (
                f"{len(distance_values_m)} points, "
                f"step {float(distance_values_m[1] - distance_values_m[0]):.0f} m"
            ),
        },
    ]
)

display(study_definition_view)


,study,varied quantity,fixed quantity,domain,sampling
0,Rate sweep,Required rate,Distance = 400 m,1 to 300 Mbps,"100 points, step 3.0 Mbps"
1,Distance sweep,Link distance,Required rate = 250 Mbps,50 to 500 m,"19 points, step 25 m"


## 3. Shared Scenario Context

Before running the two sweeps, it helps to make the common search domain visible once. The next tables summarize the reference deployment, the discrete search space inherited from the shared single-user study layer, and the PA catalog available inside that space.

These tables are structural context, not the optimization results themselves. They explain what the solver is allowed to vary later when it chooses one power-efficient operating point for each demand point.


In [3]:
reference_case_view = pd.DataFrame(
    [
        {
            "reference distance (m)": float(reference_scenario.request.distance_m),
            "reference required rate (Mbps)": float(reference_scenario.request.required_rate_bps) / 1e6,
            "reference path loss (dB)": float(reference_scenario.context.deployment.path_loss_db),
            "frame slots": int(reference_scenario.context.deployment.n_slots_win),
            "PA models": len(pa_characteristics),
        }
    ]
)

search_domain_view = pd.DataFrame(
    [
        {
            "bandwidth options (MHz)": ", ".join(
                f"{float(value) / 1e6:.0f}" for value in reference_candidate_space_row["bandwidth_options_hz"]
            ),
            "layer domain": (
                f"{int(reference_candidate_space_row['layer_domain'][0])} to "
                f"{int(reference_candidate_space_row['layer_domain'][1])}"
            ),
            "MCS domain": (
                f"{int(reference_candidate_space_row['mcs_domain'][0])} to "
                f"{int(reference_candidate_space_row['mcs_domain'][1])}"
            ),
            "slot domain": (
                f"{int(reference_candidate_space_row['slot_domain'][0])} to "
                f"{int(reference_candidate_space_row['slot_domain'][1])}"
            ),
            "PRB step": int(reference_candidate_space_row["prb_step"]),
            "raw candidates per scenario": int(reference_candidate_space_row["raw_candidate_count_total"]),
        }
    ]
)

display(reference_case_view)
display(search_domain_view)
display(pa_characteristics)


,reference distance (m),reference required rate (Mbps),reference path loss (dB),frame slots,PA models
0,400.0,1.0,129.844828,20,2


,bandwidth options (MHz),layer domain,MCS domain,slot domain,PRB step,raw candidates per scenario
0,"50, 100",1 to 4,0 to 28,1 to 20,5,389760


,pa_id,scenario_label,pa_name,source_csv,n_curve_points,p_max_w,p_idle_w,eta_max,g_pa_eff_linear,g_pa_eff_db,kappa_distortion,backoff_db
0,0,8W PA,Bae et al. NR,C:\Users\henry\Documents\Masters Thesis\code\P...,64,7.928899,5.239623,0.418165,1817.545759,32.594854,0.08,6.0
1,1,4W PA,QPA9942,C:\Users\henry\Documents\Masters Thesis\code\P...,64,1.784671,0.641172,0.352236,935.848316,29.712055,0.08,6.0


## 4. Run the Two Frontier Studies

The next cell executes both sweeps using the shared single-user study layer. This is still setup for the reader rather than the main results discussion.

The point here is to show, side by side, what has been solved in each study and how large the resulting optimization tables are. The interpretation of the frontier shapes is deferred to the later results section, where the rate and distance scenarios will be plotted next to each other in grouped visual passes.


In [4]:
rate_study = run_rate_study(
    distance_m=float(rate_study_config["distance_m"]),
    rate_targets_bps=rate_study_config["rate_targets_bps"],
    outer_parallel=NOTEBOOK_OUTER_PARALLEL,
    max_workers=NOTEBOOK_MAX_WORKERS,
)
distance_study = run_distance_study(
    distance_values_m=distance_study_config["distance_values_m"],
    required_rate_bps=float(distance_study_config["rate_target_bps"]),
    outer_parallel=NOTEBOOK_OUTER_PARALLEL,
    max_workers=NOTEBOOK_MAX_WORKERS,
)

rate_frontier_table = rate_study.frontier_table.copy()
rate_explanatory_configs = rate_study.explanatory_configs.copy()
distance_frontier_table = distance_study.frontier_table.copy()
distance_explanatory_configs = distance_study.explanatory_configs.copy()
pa_characteristics = rate_study.pa_characteristics.copy()

study_search_summary_view = pd.DataFrame(
    [
        {
            "study": "Rate sweep",
            "scenario points": int(rate_study.candidate_space_view.iloc[0]["raw_candidate_count_across_scenarios"] // rate_study.candidate_space_view.iloc[0]["raw_candidate_count_total"]),
            "PA models": len(rate_study.candidate_space_view.iloc[0]["pa_labels"]),
            "raw candidates per scenario": int(rate_study.candidate_space_view.iloc[0]["raw_candidate_count_total"]),
            "raw candidates across sweep": int(rate_study.candidate_space_view.iloc[0]["raw_candidate_count_across_scenarios"]),
            "per-PA raw counts": ", ".join(
                f"{label}: {count:,}" for label, count in rate_study.candidate_space_view.iloc[0]["raw_candidate_count_per_pa"]
            ),
        },
        {
            "study": "Distance sweep",
            "scenario points": int(distance_study.candidate_space_view.iloc[0]["raw_candidate_count_across_scenarios"] // distance_study.candidate_space_view.iloc[0]["raw_candidate_count_total"]),
            "PA models": len(distance_study.candidate_space_view.iloc[0]["pa_labels"]),
            "raw candidates per scenario": int(distance_study.candidate_space_view.iloc[0]["raw_candidate_count_total"]),
            "raw candidates across sweep": int(distance_study.candidate_space_view.iloc[0]["raw_candidate_count_across_scenarios"]),
            "per-PA raw counts": ", ".join(
                f"{label}: {count:,}" for label, count in distance_study.candidate_space_view.iloc[0]["raw_candidate_count_per_pa"]
            ),
        },
    ]
)

study_output_summary_view = pd.DataFrame(
    [
        {
            "study": "Rate sweep",
            "scenario axis": "Required rate at fixed distance",
            "frontier rows": len(rate_frontier_table),
            "explanatory rows": len(rate_explanatory_configs),
            "PA labels": ", ".join(sorted(rate_frontier_table["pa_name"].dropna().unique())),
            "x-domain": (
                f"{float(np.min(rate_frontier_table['rate_target_bps'])) / 1e6:.0f} to "
                f"{float(np.max(rate_frontier_table['rate_target_bps'])) / 1e6:.0f} Mbps"
            ),
        },
        {
            "study": "Distance sweep",
            "scenario axis": "Distance at fixed required rate",
            "frontier rows": len(distance_frontier_table),
            "explanatory rows": len(distance_explanatory_configs),
            "PA labels": ", ".join(sorted(distance_frontier_table["pa_name"].dropna().unique())),
            "x-domain": (
                f"{float(np.min(distance_frontier_table['distance_m'])):.0f} to "
                f"{float(np.max(distance_frontier_table['distance_m'])):.0f} m"
            ),
        },
    ]
)

display(study_search_summary_view)
display(study_output_summary_view)


TypeError: cannot pickle 'mappingproxy' object

## 5. PA Characteristics


In [ ]:
fig_pa_sanity, axes = plt.subplots(1, 3, figsize=(18, 5))

pout_min_dbm, pout_max_dbm = np.inf, -np.inf
pa_curve_table = build_single_user_pa_curve_table(reference_scenario)

for pa_name, pa_curve_rows in pa_curve_table.groupby("pa_name", sort=True):
    pout_w = pa_curve_rows["pout_w"].to_numpy(dtype=float)
    pin_w = pa_curve_rows["pin_w"].to_numpy(dtype=float)
    pdc_w = pa_curve_rows["pdc_w"].to_numpy(dtype=float)

    pout_dbm = 10 * np.log10(pout_w * 1000.0)
    pin_dbm = 10 * np.log10(pin_w * 1000.0)
    pdc_dbm = 10 * np.log10(pdc_w * 1000.0)

    gain_db = pout_dbm - pin_dbm
    pae_percent = 100.0 * (pout_w - pin_w) / pdc_w

    pout_min_dbm = min(pout_min_dbm, np.min(pout_dbm))
    pout_max_dbm = max(pout_max_dbm, np.max(pout_dbm))

    axes[0].plot(
        pout_dbm,
        gain_db,
        marker="o",
        label=pa_name,
    )
    axes[1].plot(
        pout_dbm,
        pae_percent,
        marker="o",
        label=pa_name,
    )
    axes[2].plot(
        pout_dbm,
        pdc_dbm,
        marker="o",
        label=pa_name,
    )

axes[0].set_title("Gain vs PA Average Output Power")
axes[1].set_title("PAE vs PA Average Output Power")
axes[2].set_title("PA DC Input Power vs PA Average Output Power")

for ax in axes:
    ax.set_xlabel("PA Average Output Power (dBm)")
    ax.set_xlim(10, 40)
    ax.grid(True, alpha=0.3)
    ax.legend()

axes[0].set_ylabel("Gain (dB)")
axes[1].set_ylabel("PAE (%)")
axes[2].set_ylabel("PA DC Input Power (dBm)")

plt.tight_layout()
plt.show()


## 6. Frontier Plots

This section now develops the frontier story in four passes. Each figure shows the rate sweep and the distance sweep together so the reader can compare how the same optimization logic responds to two different kinds of demand change.

The ordering matters. The first figure shows the core mismatch between objective and transmit output power. The middle figures then explain that mismatch through the chosen PHY resource allocations. The final figure closes the loop with the required SNR burden and an energy-per-bit view that broadens the interpretation beyond raw power minimization.


In [ ]:
%matplotlib inline

pa_label_map = pa_characteristics.set_index("pa_id")["pa_name"].to_dict()
marker_sequence = ["x", "o", "^", "D", "v", "P", "*", "+", ".", "s"]


def build_style_maps(pa_ids):
    size_max = 50
    size_min = 20
    sizes = np.linspace(size_max, size_min, len(pa_ids))
    pa_marker_map = {pa_id: marker_sequence[i % len(marker_sequence)] for i, pa_id in enumerate(pa_ids)}
    pa_size_map = {pa_id: sizes[i] for i, pa_id in enumerate(pa_ids)}
    return pa_marker_map, pa_size_map


def prep(df, x_col):
    return df.sort_values(x_col)


def req_snr_db(series):
    gamma = np.asarray(pd.to_numeric(series, errors="coerce"), dtype=float)
    return 10.0 * np.log10(np.clip(gamma, 1e-12, None))


def energy_per_bit(df):
    return df["p_dc_avg_total_w"] / np.clip(df["rate_target_bps"], 1e-12, None)


def round_up_to_step(value, step):
    value = float(value)
    step = float(step)
    if value <= 0.0:
        return step
    return step * np.ceil(value / step)


def build_axis_from_series(values, tick_step, upper_round_step, axis_min=0.0):
    values = np.asarray(pd.to_numeric(values, errors="coerce"), dtype=float)
    finite_values = values[np.isfinite(values)]
    if finite_values.size == 0:
        axis_upper = axis_min + float(upper_round_step)
    else:
        axis_upper = round_up_to_step(max(float(finite_values.max()), float(axis_min)), upper_round_step)
    tick_values = np.arange(float(axis_min), axis_upper + 0.5 * float(tick_step), float(tick_step))
    return tick_values, (float(axis_min), float(axis_upper))


def build_integer_plot_domain(values, *, max_ticks=8):
    ordered_values = sorted({int(value) for value in values})
    stride = max(1, int(np.ceil(len(ordered_values) / int(max_ticks))))
    tick_values = ordered_values[::stride]
    if tick_values[-1] != ordered_values[-1]:
        tick_values.append(ordered_values[-1])

    padding = 0.5 if len(ordered_values) == 1 else min(np.diff(ordered_values)) / 2.0
    return {
        "ticks": np.asarray(tick_values, dtype=float),
        "limits": (
            float(ordered_values[0] - padding),
            float(ordered_values[-1] + padding),
        ),
    }


def format_x_value(value, unit):
    if unit == "Mbps":
        return f"{float(value):.1f} Mbps"
    return f"{float(value):.0f} m"


reference_candidate_space_row = reference_candidate_space_view.iloc[0]
prb_step = int(reference_candidate_space_row["prb_step"])
prb_upper = max(
    int(prb_max_bwp)
    for _scenario_label, _bwp_index, prb_max_bwp in reference_candidate_space_row["max_prbs_by_bwp"]
)
FRAME_SLOT_COUNT = int(reference_candidate_space_row["slot_domain"][1])
SUMMARY_PLOT_DOMAINS = {
    "layers": build_integer_plot_domain(reference_candidate_space_row["layer_domain"]),
    "mcs": build_integer_plot_domain(reference_candidate_space_row["mcs_domain"]),
    "n_prb": build_integer_plot_domain(range(prb_step, prb_upper + prb_step, prb_step)),
    "n_slots_on": build_integer_plot_domain(
        range(
            int(reference_candidate_space_row["slot_domain"][0]),
            int(reference_candidate_space_row["slot_domain"][1]) + 1,
        )
    ),
}

rate_plot_distance_m = float(rate_study_config["distance_m"])
plot_rate_frontier = rate_frontier_table.assign(rate_target_mbps=rate_frontier_table["rate_target_bps"] / 1e6).copy()
plot_distance_frontier = distance_frontier_table.copy()

distance_fixed_rate_mbps = float(distance_study_config["rate_target_bps"]) / 1e6
rate_tick_values, rate_limits = build_axis_from_series(
    np.asarray(rate_study_config["rate_targets_bps"], dtype=float) / 1e6,
    tick_step=50.0,
    upper_round_step=100.0,
    axis_min=0.0,
)
distance_tick_values, distance_limits = build_axis_from_series(
    np.asarray(distance_study_config["distance_values_m"], dtype=float),
    tick_step=50.0,
    upper_round_step=100.0,
    axis_min=0.0,
)

SCENARIO_SPECS = [
    {
        "label": "Rate sweep",
        "table": plot_rate_frontier,
        "x_col": "rate_target_mbps",
        "x_label": "Target Rate (Mbps)",
        "x_ticks": rate_tick_values,
        "x_limits": rate_limits,
        "x_unit": "Mbps",
        "subtitle": f"Fixed distance: {int(rate_plot_distance_m)} m",
    },
    {
        "label": "Distance sweep",
        "table": plot_distance_frontier,
        "x_col": "distance_m",
        "x_label": "Distance (m)",
        "x_ticks": distance_tick_values,
        "x_limits": distance_limits,
        "x_unit": "m",
        "subtitle": f"Fixed rate: {distance_fixed_rate_mbps:.0f} Mbps",
    },
]

METRIC_SPECS = {
    "p_dc_avg_total_w": {
        "label": "Frame-averaged total PA DC input power (W)",
        "style": "line",
    },
    "p_out_total_w": {
        "label": "Total PA output power (W)",
        "style": "line",
    },
    "layers": {
        "label": "Layers",
        "style": "scatter",
        "domain_key": "layers",
    },
    "mcs": {
        "label": "MCS index",
        "style": "scatter",
        "domain_key": "mcs",
    },
    "n_prb": {
        "label": "Allocated PRBs",
        "style": "scatter",
        "domain_key": "n_prb",
    },
    "n_slots_on": {
        "label": f"Allocated slots per {FRAME_SLOT_COUNT}-slot frame",
        "style": "scatter",
        "domain_key": "n_slots_on",
    },
    "gamma_req_db": {
        "label": "Required SNR (dB)",
        "style": "line",
    },
    "energy_per_bit": {
        "label": "PA energy per delivered bit (J/bit)",
        "style": "line",
        "yscale": "log",
    },
}


def extract_metric_series(df, metric_key):
    if metric_key == "gamma_req_db":
        return req_snr_db(df["gamma_req_lin"])
    if metric_key == "energy_per_bit":
        return np.asarray(energy_per_bit(df), dtype=float)
    return np.asarray(pd.to_numeric(df[metric_key], errors="coerce"), dtype=float)


def build_winner_curve(table, x_col):
    tie_break_cols = [
        "p_dc_avg_total_w",
        "bandwidth_hz",
        "n_prb",
        "n_slots_on",
        "layers",
        "mcs",
        "pa_id",
    ]
    return (
        table.sort_values([x_col] + tie_break_cols)
        .groupby(x_col, as_index=False)
        .first()
        .reset_index(drop=True)
    )


def plot_metric_panel(ax, scenario_spec, metric_key):
    metric_spec = METRIC_SPECS[metric_key]
    frontier_table = scenario_spec["table"]
    x_col = scenario_spec["x_col"]
    pa_ids = sorted(frontier_table["pa_id"].unique())
    pa_marker_map, pa_size_map = build_style_maps(pa_ids)

    for pa_id in pa_ids:
        df = prep(frontier_table[frontier_table["pa_id"] == pa_id], x_col)
        y_values = extract_metric_series(df, metric_key)
        if metric_spec["style"] == "scatter":
            ax.scatter(
                df[x_col],
                y_values,
                marker=pa_marker_map[pa_id],
                s=pa_size_map[pa_id],
                label=pa_label_map.get(pa_id, f"PA{pa_id}"),
            )
        else:
            ax.plot(df[x_col], y_values, label=pa_label_map.get(pa_id, f"PA{pa_id}"))

    ax.set_xlabel(scenario_spec["x_label"])
    ax.set_ylabel(metric_spec["label"])
    ax.set_xticks(scenario_spec["x_ticks"])
    ax.set_xlim(*scenario_spec["x_limits"])
    ax.grid(True, alpha=0.3)

    domain_key = metric_spec.get("domain_key")
    if domain_key is not None:
        axis_config = SUMMARY_PLOT_DOMAINS[domain_key]
        ax.set_ylim(*axis_config["limits"])
        ax.set_yticks(axis_config["ticks"])

    yscale = metric_spec.get("yscale")
    if yscale is not None:
        ax.set_yscale(yscale)

    ax.set_title(f"{scenario_spec['label']}: {metric_spec['label']}\n{scenario_spec['subtitle']}")


def plot_grouped_frontier_figure(metric_keys, figure_title):
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(figure_title, fontsize=14)

    for row_idx, scenario_spec in enumerate(SCENARIO_SPECS):
        for col_idx, metric_key in enumerate(metric_keys):
            plot_metric_panel(axes[row_idx, col_idx], scenario_spec, metric_key)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=max(1, len(labels)), bbox_to_anchor=(0.5, 0.975))
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()
    return fig


def build_objective_output_summary():
    rows = []
    for scenario_spec in SCENARIO_SPECS:
        winner_curve = build_winner_curve(scenario_spec["table"], scenario_spec["x_col"])
        start_row = winner_curve.iloc[0]
        end_row = winner_curve.iloc[-1]
        rows.append(
            {
                "scenario": scenario_spec["label"],
                "start point": format_x_value(start_row[scenario_spec["x_col"]], scenario_spec["x_unit"]),
                "end point": format_x_value(end_row[scenario_spec["x_col"]], scenario_spec["x_unit"]),
                "winner changes": int((winner_curve["pa_id"] != winner_curve["pa_id"].shift()).sum() - 1),
                "PA DC growth x": float(end_row["p_dc_avg_total_w"] / start_row["p_dc_avg_total_w"]),
                "PA output growth x": float(end_row["p_out_total_w"] / start_row["p_out_total_w"]),
            }
        )
    return pd.DataFrame(rows)


def build_regime_summary(metric_keys, *, max_rows_per_scenario=6):
    rows = []
    for scenario_spec in SCENARIO_SPECS:
        winner_curve = build_winner_curve(scenario_spec["table"], scenario_spec["x_col"])
        previous_row = None
        shown_rows = 0
        for _, current_row in winner_curve.iterrows():
            if previous_row is None:
                previous_row = current_row
                continue
            if any(int(current_row[key]) != int(previous_row[key]) for key in metric_keys):
                row = {
                    "scenario": scenario_spec["label"],
                    "at": format_x_value(current_row[scenario_spec["x_col"]], scenario_spec["x_unit"]),
                    "winner PA": str(current_row["pa_name"]),
                }
                for key in metric_keys:
                    row[METRIC_SPECS[key]["label"]] = int(current_row[key])
                rows.append(row)
                shown_rows += 1
                if shown_rows >= max_rows_per_scenario:
                    break
            previous_row = current_row
    return pd.DataFrame(rows)


def build_physical_energy_summary():
    rows = []
    for scenario_spec in SCENARIO_SPECS:
        winner_curve = build_winner_curve(scenario_spec["table"], scenario_spec["x_col"])
        energy_values = np.asarray(energy_per_bit(winner_curve), dtype=float)
        snr_values = req_snr_db(winner_curve["gamma_req_lin"])
        min_energy_idx = int(np.argmin(energy_values))
        max_snr_idx = int(np.argmax(snr_values))
        min_energy_row = winner_curve.iloc[min_energy_idx]
        max_snr_row = winner_curve.iloc[max_snr_idx]
        rows.append(
            {
                "scenario": scenario_spec["label"],
                "minimum energy per bit": float(energy_values[min_energy_idx]),
                "at minimum energy": format_x_value(min_energy_row[scenario_spec["x_col"]], scenario_spec["x_unit"]),
                "PA at minimum energy": str(min_energy_row["pa_name"]),
                "maximum required SNR (dB)": float(snr_values[max_snr_idx]),
                "at maximum required SNR": format_x_value(max_snr_row[scenario_spec["x_col"]], scenario_spec["x_unit"]),
            }
        )
    return pd.DataFrame(rows)


### 6.1 Objective vs Output Power

This first figure establishes the central observation of the section. The left-hand panels show the optimization objective itself, namely frame-averaged total PA DC input power. The right-hand panels then show the corresponding total PA output power for the same frontier rows.

Read these two columns together. The point is to see that the optimal PA DC power rises with demand in both studies, but it does not increase in simple proportion to the radiated output power. That mismatch is the signal that the optimizer is changing *how* it serves the user, not only *how hard* it drives the transmitter.


In [ ]:
fig_objective_vs_output = plot_grouped_frontier_figure(
    metric_keys=["p_dc_avg_total_w", "p_out_total_w"],
    figure_title="Objective vs Output Power Across the Two Single-User Sweeps",
)
objective_output_summary = build_objective_output_summary()
display(objective_output_summary)


The summary table under the figure should be read as a compact numerical version of the same visual point. For each scenario it reports the start and end of the winning frontier and compares how much the objective grows relative to total output power. If those growth factors differ, then the optimum is being shaped by resource allocation choices and PA behavior, not only by required radiated power.


### 6.2 Spatial / Coding Choices

The next figure begins to explain the earlier gap. These plots show the chosen number of spatial layers and the selected MCS index along the frontier for both studies.

This is where the notebook should make it obvious that the optimizer does not always chase a more aggressive coding point. Sometimes it backs off on MCS and compensates with more structural resource use because that combination yields a lower frame-averaged PA DC cost.


In [ ]:
fig_spatial_coding = plot_grouped_frontier_figure(
    metric_keys=["layers", "mcs"],
    figure_title="Spatial / Coding Choices Along the Winning Frontier",
)
spatial_coding_summary = build_regime_summary(["layers", "mcs"])
display(spatial_coding_summary)


The regime-change table highlights only the points where the winning configuration actually changes its structural choice. That is the most useful way to read these panels: not as every row individually, but as a sequence of operating regimes where the optimizer decides that a different layers-and-MCS combination is now cheaper.


### 6.3 Time / Frequency Allocation

The third figure continues the same explanation, but now in the time-frequency dimensions. The PRB and slot panels show how the optimizer uses bandwidth occupancy and time occupancy to control the power cost of meeting a harder user requirement.

These panels are the clearest view of duty cycling. Early in the sweep there is often room to trade more scheduling resource for a lower instantaneous burden. Later in the sweep that flexibility shrinks, and the objective begins to move more closely with the required output power.


In [ ]:
fig_time_frequency = plot_grouped_frontier_figure(
    metric_keys=["n_prb", "n_slots_on"],
    figure_title="Time / Frequency Allocation Along the Winning Frontier",
)
time_frequency_summary = build_regime_summary(["n_prb", "n_slots_on"])
display(time_frequency_summary)


Again, the table is there to mark the regime boundaries rather than to replace the figure. It shows where the winner starts using more PRBs, more slots, or both, and makes the time-frequency duty-cycling logic visible in data rather than only in prose.


### 6.4 Physical Requirement vs Energy Metric

The last grouped figure closes the single-user interpretation. Required SNR shows the physical burden implied by the winning operating points, while PA energy per delivered bit gives a broader system-efficiency lens than raw frame-averaged power alone.

This final pair should be read carefully. Required SNR helps explain *why* the optimizer must move toward costlier operating points, while energy per bit reminds the reader that minimizing frame-averaged PA DC power is not the only way one might compare configurations.


In [ ]:
fig_physical_energy = plot_grouped_frontier_figure(
    metric_keys=["gamma_req_db", "energy_per_bit"],
    figure_title="Physical Requirement vs Energy Metric Along the Winning Frontier",
)
physical_energy_summary = build_physical_energy_summary()
display(physical_energy_summary)


The summary table identifies where the winning frontier attains its lowest PA energy-per-bit point and where the required SNR becomes most severe. Together with the earlier figures, this closes the single-user story: the optimum is demand-dependent, PA-dependent, and resource-allocation-dependent.


### 6.5 Bridge to Multi-User Scheduling

This single-user study has shown that there is no one static operating point that deserves to be called optimal in every case. The best configuration changes with required throughput, with distance, and with the structural choices the scheduler is allowed to make.

That observation is exactly why the next problem is more difficult. In a real system, many users must be scheduled at once, and each user can bring its own locally attractive operating points. The next step is therefore not to choose one single-user frontier row in isolation, but to coordinate many such rows under a shared multi-user TDMA resource budget.


## 7. Export Datasets


In [ ]:
from pathlib import Path

required_objects = [
    "fig_pa_sanity",
    "fig_objective_vs_output",
    "fig_spatial_coding",
    "fig_time_frequency",
    "fig_physical_energy",
    "objective_output_summary",
    "spatial_coding_summary",
    "time_frequency_summary",
    "physical_energy_summary",
    "rate_frontier_table",
    "distance_frontier_table",
    "pa_characteristics",
    "rate_explanatory_configs",
    "distance_explanatory_configs",
]
missing = [name for name in required_objects if name not in globals()]
assert not missing, f"Run the exploration and plot sections before saving outputs. Missing: {missing}"

graphs_dir = Path("graphs")
csv_dir = Path("csvs")
graphs_dir.mkdir(parents=True, exist_ok=True)
csv_dir.mkdir(parents=True, exist_ok=True)

image_targets = {
    "pa_characteristics.png": fig_pa_sanity,
    "frontier_objective_vs_output.png": fig_objective_vs_output,
    "frontier_spatial_coding.png": fig_spatial_coding,
    "frontier_time_frequency.png": fig_time_frequency,
    "frontier_physical_energy.png": fig_physical_energy,
}

for filename, fig in image_targets.items():
    fig.savefig(graphs_dir / filename, dpi=300, bbox_inches="tight")

table_targets = {
    "rate_frontier_by_pa.csv": rate_frontier_table,
    "distance_frontier_by_pa.csv": distance_frontier_table,
    "pa_characteristics.csv": pa_characteristics,
    "rate_explanatory_configs.csv": rate_explanatory_configs,
    "distance_explanatory_configs.csv": distance_explanatory_configs,
    "objective_output_summary.csv": objective_output_summary,
    "spatial_coding_summary.csv": spatial_coding_summary,
    "time_frequency_summary.csv": time_frequency_summary,
    "physical_energy_summary.csv": physical_energy_summary,
}

for filename, df in table_targets.items():
    df.to_csv(csv_dir / filename, index=False)

print(f"Saved {len(image_targets)} images to {graphs_dir.resolve()}")
print(f"Saved {len(table_targets)} tables to {csv_dir.resolve()}")
